In [1]:
# %%
import gc
from pathlib import Path

import polars as pl
from tqdm.auto import tqdm


INPUT_DIR = Path("data/preprocessed_dataset")

OUTPUT_DIRS = {
    "train": Path("data/final_train"),
    "val": Path("data/final_val"),
    "test": Path("data/final_test"),
}

ID_COL = "ид_канала_данных"

TARGET = "тревожное"
TARGET_UPPER_BOUND = 48

RANDOM_STATE = 69

TRAIN_SIZE = 0.9
VAL_SIZE = 0.05
TEST_SIZE = 0.05


def make_target(lf: pl.LazyFrame) -> pl.LazyFrame:
    return (
        lf
        .with_columns(
            pl.col(TARGET)
            .fill_null(TARGET_UPPER_BOUND)
            .clip(upper_bound=TARGET_UPPER_BOUND)
            .alias("target_lower"),

            pl.when(pl.col(TARGET) <= TARGET_UPPER_BOUND)
            .then(pl.col(TARGET))
            .otherwise(-1.0)
            .alias("target_upper"),
        )
        .drop(TARGET)
    )


def get_device_split(files: list[Path]) -> pl.DataFrame:
    ids = (
        pl.scan_parquet(files)
        .select(ID_COL)
        .unique()
        .sort(ID_COL)
        .collect()
        .sample(
            fraction=1.0,
            shuffle=True,
            seed=RANDOM_STATE,
        )
    )

    n = ids.height

    n_train = int(n * TRAIN_SIZE)
    n_val = int(n * VAL_SIZE)

    train_ids = (
        ids
        .head(n_train)
        .with_columns(
            pl.lit("train").alias("__split")
        )
    )

    val_ids = (
        ids
        .slice(n_train, n_val)
        .with_columns(
            pl.lit("val").alias("__split")
        )
    )

    test_ids = (
        ids
        .slice(n_train + n_val)
        .with_columns(
            pl.lit("test").alias("__split")
        )
    )

    return pl.concat(
        [
            train_ids,
            val_ids,
            test_ids,
        ]
    )


def build_final_datasets() -> None:
    if abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) > 1e-9:
        raise ValueError("TRAIN_SIZE + VAL_SIZE + TEST_SIZE must equal 1")

    for output_dir in OUTPUT_DIRS.values():
        output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

    files = sorted(
        INPUT_DIR.glob("*.parquet")
    )

    device_split = get_device_split(files)

    print(
        device_split
        .group_by("__split")
        .len()
        .with_columns(
            (
                pl.col("len")
                / pl.col("len").sum()
            ).alias("share")
        )
        .sort("__split")
    )

    split_lf = device_split.lazy()

    for file_path in tqdm(
        files,
        desc="Processing chunks",
    ):
        lf = (
            pl.scan_parquet(file_path)
            .pipe(make_target)
            .join(
                split_lf,
                on=ID_COL,
                how="left",
            )
        )

        for split_name, output_dir in OUTPUT_DIRS.items():
            (
                lf
                .filter(
                    pl.col("__split") == split_name
                )
                .drop("__split")
                .sink_parquet(
                    output_dir / file_path.name
                )
            )

        del lf
        gc.collect()


build_final_datasets()

/home/arseniy/miniconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


shape: (3, 3)
┌─────────┬───────┬──────────┐
│ __split ┆ len   ┆ share    │
│ ---     ┆ ---   ┆ ---      │
│ str     ┆ u32   ┆ f64      │
╞═════════╪═══════╪══════════╡
│ test    ┆ 577   ┆ 0.050096 │
│ train   ┆ 10366 ┆ 0.899983 │
│ val     ┆ 575   ┆ 0.049922 │
└─────────┴───────┴──────────┘


Processing chunks: 100%|██████████| 116/116 [02:28<00:00,  1.28s/it]


In [2]:
print(f'{pl.scan_parquet("data/final_train/*.parquet").select(pl.len()).collect()[0,0]=:_}')
print(f'{pl.scan_parquet("data/final_test/*.parquet").select(pl.len()).collect()[0,0]=:_}')
print(f'{pl.scan_parquet("data/final_val/*.parquet").select(pl.len()).collect()[0,0]=:_}')

pl.scan_parquet("data/final_train/*.parquet").select(pl.len()).collect()[0,0]=115_450_010
pl.scan_parquet("data/final_test/*.parquet").select(pl.len()).collect()[0,0]=12_072_037
pl.scan_parquet("data/final_val/*.parquet").select(pl.len()).collect()[0,0]=7_739_021


In [3]:
print(f'{pl.scan_parquet("data/final_train/*.parquet").select(pl.col("ид_канала_данных").n_unique()).collect()[0,0]=:_}')
print(f'{pl.scan_parquet("data/final_test/*.parquet").select(pl.col("ид_канала_данных").n_unique()).collect()[0,0]=:_}')
print(f'{pl.scan_parquet("data/final_val/*.parquet").select(pl.col("ид_канала_данных").n_unique()).collect()[0,0]=:_}')

pl.scan_parquet("data/final_train/*.parquet").select(pl.col("ид_канала_данных").n_unique()).collect()[0,0]=10_366
pl.scan_parquet("data/final_test/*.parquet").select(pl.col("ид_канала_данных").n_unique()).collect()[0,0]=577
pl.scan_parquet("data/final_val/*.parquet").select(pl.col("ид_канала_данных").n_unique()).collect()[0,0]=575
